In [1]:
from functools import wraps
from typing import Callable, Any, TypeVar, ParamSpec, Literal, Union
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from loguru import logger
from huggingface_hub import hf_hub_download
import polars as pl

import os
from dotenv import load_dotenv
load_dotenv()
KAGGLE_USERNAME = os.getenv("KAGGLE_USERNAME")
KAGGLE_KEY = os.getenv("KAGGLE_KEY")

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

/home/user/datacrawling/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logger.add(
    "log.json", serialize=True, rotation="10 MB", retention="10 days", level="DEBUG"
)
logger.add("error_log.log", level="ERROR")

2

In [3]:
P = ParamSpec("P")
R = TypeVar("R")


def logger_wrapper(func: Callable[P, R]) -> Callable[P, R]:
    @wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        with logger.catch():
            return func(*args, **kwargs)

    return wrapper

## my code

In [ ]:
class BaseDownloader(ABC):
    def __init__(self, url: str, id: str, path: str, *args, **kwargs) -> None:
        self.url = url
        self.path = path
        self.id = id
        self.args = args
        self.kwargs = kwargs

        # from pathlib import Path
        # Path(self.path).mkdir(parents=True, exist_ok=True)

    @abstractmethod
    def _download(self) -> None:
        """Download data from url."""
        pass

    @abstractmethod
    def _validate(self) -> None:
        """Validate arguments."""
        pass

    @logger_wrapper
    def execute(self) -> None:
        """Execute dowload progress."""
        logger.info(
            f"[{self.__class__.__name__}] Downloading data:\n"
            f"- url: '{self.url}'\n"
            f"- id: '{self.id}'\n"
            f"- destination: '{self.path}'"
        )
        self._validate()
        self._download()
        logger.success(f"[{self.__class__.__name__}] Complete downloading data.")


In [ ]:
import gdown


class GoogleDriverDownloader(BaseDownloader):
    def __init__(
        self,
        url: str = "",
        path: str = "",
        id: str = None,
        item_type: Literal["file", "folder"] = "",
        *args,
        **kwargs,
    ) -> None:
        super().__init__(url, id, path, *args, **kwargs)
        self.item_type = item_type

    def _validate(self) -> None:
        if not self.path:
            raise ValueError("Need provide path for download.")
        if self.item_type is None:
            raise ValueError("type is required")
        if self.item_type not in ("file", "folder"):
            raise ValueError("type must be 'file' or 'folder'")
        if self.item_type == "folder":
            if not self.url and not self.id:
                raise ValueError("Need provide url or id for folder download.")
        else:
            if not self.url:
                raise ValueError("Need provide url for file download.")

    def _download_folder(self) -> None:
        if self.url:
            gdown.download_folder(
                url=self.url, output=self.path, *self.args, **self.kwargs
            )
            return
        if self.id:
            gdown.download_folder(
                id=self.id, output=self.path, *self.args, **self.kwargs
            )
            return

    def _download_file(self) -> None:
        gdown.download(url=self.url, output=self.path, *self.args, **self.kwargs)

    def _download(self) -> None:
        if self.item_type == "folder":
            # self._download_folder()
            pass
        else:
            self._download_file()


gg_downloader = GoogleDriverDownloader(
    id="14vCUXklVCUN_rCPKYN7ys2tDJG72SNYw",
    path="./test_data",
    item_type="folder",
    quiet=False,
)

gg_downloader.execute()

In [ ]:


class KaggleDownloader(BaseDownloader):
    def __init__(self, url: str = "", id: str = "", path: str = "", *args, **kwargs) -> None:
        super().__init__(url, id, path, *args, **kwargs)
    
    def _validate(self) -> None:
        if not self.id:
            raise ValueError("Need provide id for kaggle dataset.")
        if not self.path:
            raise ValueError("Need provide path for download.")

    def _download(self) -> None:
        api.dataset_download_files(self.id, path=self.path, **self.kwargs)

KaggleDownloader(id="muratkokludataset/acoustic-extinguisher-fire-dataset", path="./test_data/", unzip=True).execute()

In [ ]:


class HuggingFaceDownloader(BaseDownloader):
    def _validate(self) -> None:
        if not self.id:
            raise ValueError("id is required")

    def _download(self) -> None:
        file_path = hf_hub_download(
            repo_id=self.id, 
            filename=self.kwargs.get("file_name"), 
            repo_type="dataset"
        )

        pl.scan_parquet(file_path).collect().write_parquet(self.path)
HuggingFaceDownloader(
    url="",
    id="tmquan/sapnhap-bando-vn",
    path="./data/sapnhap-bando-vn.parquet",
    file_name="data/all.parquet"
).execute()

## try claude code

In [4]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from loguru import logger


# ── Config per source ──────────────────────────────────────────────
@dataclass
class DownloadConfig:
    url: str = ""
    path: str = ""
    id: str = ""
    extra: dict = field(default_factory=dict)

# ── Base: Template Method ──────────────────────────────────────────
class BaseDownloader(ABC):
    def __init__(self, config: DownloadConfig) -> None:
        self.config = config

    @abstractmethod
    def _validate(self) -> None: ...

    @abstractmethod
    def _download(self) -> None: ...

    @logger_wrapper
    def execute(self) -> None:
        """Fixed template: validate → download → log."""
        logger.info(f"[{self.__class__.__name__}] Starting download...")
        self._validate()
        self._download()
        logger.success(f"[{self.__class__.__name__}] Done.")

In [5]:
# ── Google Drive ───────────────────────────────────────────────────
class GDriveDownloader(BaseDownloader):
    def _validate(self) -> None:
        item_type = self.config.get("extra", {}).get("item_type")
        if item_type not in ("file", "folder"):
            raise ValueError("'item_type' must be 'file' or 'folder'")
        if item_type == "folder" and not self.config.get("url") and not self.config.get("id"):
            raise ValueError("Provide url or id for folder download.")
        if item_type == "file" and not self.config.get("url"):
            raise ValueError("Provide url for file download.")

    def _download(self) -> None:
        import gdown
        item_type = self.config.get("extra", {}).get("item_type")
        kwargs = {k: v for k, v in self.config.get("extra", {}).items() if k != "item_type"}

        if item_type == "folder":
            if self.config.get("url"):
                gdown.download_folder(
                    url=self.config.get("url"),
                    output=self.config.get("path"),
                    **kwargs,
                )
            if self.config.get("id"):
                gdown.download_folder(
                    id=self.config.get("id"),
                    output=self.config.get("path"),
                    **kwargs,
                )
        else:
            gdown.download(
                url=self.config.get("url"),
                output=self.config.get("path"),
                **kwargs,
            )
gg_driver_config = {
    "url": "",
    "id": "14vCUXklVCUN_rCPKYN7ys2tDJG72SNYw",
    "path": "/home/user/datacrawling/data/raw/2026-05-12/test_data",
    "extra": {
        "quiet": False,
        "item_type": "folder",
    }
    
}
# GDriveDownloader(gg_driver_config).execute()

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
KAGGLE_USERNAME = os.getenv("KAGGLE_USERNAME")
KAGGLE_KEY = os.getenv("KAGGLE_KEY")

# ── Kaggle (stub) ──────────────────────────────────────────────────
class KaggleDownloader(BaseDownloader):
    def _validate(self) -> None:
        if not self.config.get("extra", {}).get("dataset"):
            raise ValueError("'dataset' is required (e.g. 'user/dataset-name')")

    def _download(self) -> None:
        import kaggle
        kaggle.api.dataset_download_files(
            self.config.get("extra", {}).get("dataset"),
            path=self.config.get("path"),
            unzip=self.config.get("extra", {}).get("unzip", True),
        )

kaggle_dowloader_config = {
    "url": "",
    "id": "",
    "path": "./test_data",
    "extra": {
        "dataset": "muratkokludataset/acoustic-extinguisher-fire-dataset",
    }
}
# KaggleDownloader(kaggle_dowloader_config).execute()

In [15]:
# ── HuggingFace (stub) ─────────────────────────────────────────────
class HuggingFaceDownloader(BaseDownloader):
    def _validate(self) -> None:
        if not self.config.get("id"):
            raise ValueError("id is required")

    def _save_to_parquet(self, file_path: str) -> None:
        pl.scan_parquet(file_path).collect().write_parquet(self.config.get("path"))

    def _download(self) -> None:
        file_path = hf_hub_download(
            repo_id=self.config.get("id"), 
            filename=self.config.get("extra", {}).get("file_name"), 
            repo_type="dataset"
        )

        self._save_to_parquet(file_path)

hugging_face_config = {
    "url": "",
    "id": "tmquan/sapnhap-bando-vn",
    "path": "/home/user/datacrawling/data/raw/2026-05-12/sapnhap-bando-vn.parquet",
    "extra": {
        "file_name": "data/all.parquet"
    }
}
# HuggingFaceDownloader(config=hugging_face_config).execute()

In [ ]:
from enum import Enum

class SourceType(str, Enum):
    GDRIVE = "gdrive"
    KAGGLE = "kaggle"
    HUGGINGFACE = "huggingface"

from typing import Type


# ── Factory (Strategy selection) ──────────────────────────────────
# dict of source -> downloader class
DOWNLOADER_MAP: dict[SourceType, Type[BaseDownloader]] = {
    SourceType.GDRIVE:       GDriveDownloader,
    SourceType.KAGGLE:       KaggleDownloader,
    SourceType.HUGGINGFACE:  HuggingFaceDownloader,
}

@logger_wrapper
def get_downloader(source: str, config: DownloadConfig) -> BaseDownloader:
    cls = DOWNLOADER_MAP.get(source)
    if cls is None:
        raise ValueError(f"Unknown source '{source}'. Choose from: {list(DOWNLOADER_MAP)}")
    return cls(config)


# ── Usage ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    # downloader = get_downloader(SourceType.HUGGINGFACE, hugging_face_config)
    # downloader.execute()
    # get_downloader(SourceType.GDRIVE, gg_driver_config).execute()
    get_downloader(SourceType.KAGGLE, kaggle_dowloader_config).execute()

In [ ]:
# factory from abstract class to concrete class creator
from abc import ABC, abstractmethod

class DownloaderCreator(ABC):

    def __init__(self, config) -> None:
        self.config = config
    
    @abstractmethod
    def create_downloader(self) -> BaseDownloader:
        pass

    def execute(self) -> None:
        downloader = self.create_downloader()
        downloader.execute()


class GDriverDownloaderCreator(DownloaderCreator):
    def create_downloader(self) -> BaseDownloader:
        return GDriveDownloader(self.config)

class KaggleDownloaderCreator(DownloaderCreator):
    def create_downloader(self) -> BaseDownloader:
        return KaggleDownloader(self.config)

class HuggingFaceDownloaderCreator(DownloaderCreator):
    def create_downloader(self) -> BaseDownloader:
        return HuggingFaceDownloader(self.config)
        

KaggleDownloaderCreator(kaggle_dowloader_config).execute()

2026-05-13 10:03:41.599 | INFO     | __main__:execute:28 - [KaggleDownloader] Starting download...


Dataset URL: https://www.kaggle.com/datasets/muratkokludataset/acoustic-extinguisher-fire-dataset


2026-05-13 10:03:43.141 | SUCCESS  | __main__:execute:31 - [KaggleDownloader] Done.


In [ ]:
# register as a class
class DownloaderFactory:

    _registry = {}

    @classmethod
    def register(cls, source, downloader_cls):
        if source in cls._registry:
            logger.warning(f"Downloader for source '{source}' already registered.")
        else:
            cls._registry[source] = downloader_cls

    @classmethod
    def create(cls, source, config):
        downloader_cls = cls.get(source)
        if downloader_cls is None:
            raise ValueError(f"Downloader for source '{source}' not found.")

        return downloader_cls(config)
    
    @classmethod
    def delete(cls, source):
        if source in cls._registry:
            del cls._registry[source]

DownloaderFactory.register("hugging_face", HuggingFaceDownloader)

dowloader_factory = DownloaderFactory()
hugging_face_downloader = dowloader_factory.create("hugging_face", hugging_face_config)
hugging_face_downloader.execute()

2026-05-13 11:28:06.821 | INFO     | __main__:execute:28 - [HuggingFaceDownloader] Starting download...
2026-05-13 11:28:07.171 | SUCCESS  | __main__:execute:31 - [HuggingFaceDownloader] Done.


In [38]:
from enum import Enum

class SourceType(str, Enum):
    GDRIVE = "gdrive"
    KAGGLE = "kaggle"
    HUGGINGFACE = "huggingface"

# register as a decorator
class DownloaderFactory2:

    _registry = {}

    @classmethod
    def register(cls, source):
        def decorator(downloader_cls):
            if source in cls._registry:
                logger.warning(f"[Factory] Overwriting existing downloader for '{source}'.")
            cls._registry[source] = downloader_cls
            return downloader_cls
        return decorator

    @classmethod
    def create(cls, source, config):
        downloader_cls = cls._registry.get(source)
        if downloader_cls is None:
            available = list(cls._registry.keys())
            raise ValueError(
                f"No downloader registered for '{source}'. "
                f"Available: {available}"
            )
        return downloader_cls(config)
    
    @classmethod
    def unregister(cls, source):
        cls._registry.pop(source, None)
    
    @classmethod
    def available(cls) -> list[str]:
        return list(cls._registry.keys())

@DownloaderFactory2.register(SourceType.KAGGLE)
class KaggleDownloader2(BaseDownloader):
    def _validate(self) -> None:
        if not self.config.get("extra", {}).get("dataset"):
            raise ValueError("'dataset' is required (e.g. 'user/dataset-name')")

    def _download(self) -> None:
        logger.info(self.__class__.__name__)

@DownloaderFactory2.register(SourceType.HUGGINGFACE)
class HuggingFaceDownloader2(BaseDownloader):
    def _validate(self) -> None:
        if not self.config.get("id"):
            raise ValueError("id is required")

    def _download(self) -> None:
        logger.info(self.__class__.__name__)

@DownloaderFactory2.register(SourceType.HUGGINGFACE)
class HuggingFaceDownloader3(BaseDownloader):
    def _validate(self) -> None:
        if not self.config.get("id"):
            raise ValueError("id is required")

    def _download(self) -> None:
        logger.info(self.__class__.__name__)

kaggle_downloader = DownloaderFactory2.create(
    SourceType.KAGGLE,
    kaggle_dowloader_config
)
kaggle_downloader.execute()

hugging_face_downloader = DownloaderFactory2.create(
    SourceType.HUGGINGFACE,
    hugging_face_config
)
hugging_face_downloader.execute()

logger.info(DownloaderFactory2.available())

2026-05-13 11:41:53.947 | WARNING  | __main__:decorator:17 - [Factory] Overwriting existing downloader for 'SourceType.HUGGINGFACE'.
2026-05-13 11:41:53.950 | INFO     | __main__:execute:28 - [KaggleDownloader2] Starting download...
2026-05-13 11:41:53.953 | INFO     | __main__:_download:48 - KaggleDownloader2
2026-05-13 11:41:53.957 | SUCCESS  | __main__:execute:31 - [KaggleDownloader2] Done.
2026-05-13 11:41:53.950 | INFO     | __main__:execute:28 - [KaggleDownloader2] Starting download...
2026-05-13 11:41:53.953 | INFO     | __main__:_download:48 - KaggleDownloader2
2026-05-13 11:41:53.957 | SUCCESS  | __main__:execute:31 - [KaggleDownloader2] Done.
2026-05-13 11:41:53.959 | INFO     | __main__:execute:28 - [HuggingFaceDownloader3] Starting download...
2026-05-13 11:41:53.963 | INFO     | __main__:_download:66 - HuggingFaceDownloader3
2026-05-13 11:41:53.965 | SUCCESS  | __main__:execute:31 - [HuggingFaceDownloader3] Done.
2026-05-13 11:41:53.968 | INFO     | __main__:<module>:80 - 